# Schema Matching Tutorial

> **Documentation:** For detailed API references and additional examples, see:
> - [Schema Matching Wiki](../../../wiki/SchemaMatching.md) - LLM-based matching, evaluators, translation
> - [Normalization Wiki](../../../wiki/Normalization.md) - Profiling, specs, transformations, validators

This tutorial demonstrates a data integration workflow:

1. **Schema Matching** - Discover column correspondences using LLM
2. **JSON Schema Integration** - Load normalization specs from JSON Schema
3. **Translation + Normalization** - Transform data to match target schema

## Use Case: Building a Movie Database

We integrate two source datasets:
- `actors.csv` - Oscar-winning actors (tab-separated, no headers)
- `movie_list.csv` - Box office data (different column names)

Target: unified DataFrame matching `target_schema.json`

In [1]:
from pathlib import Path
import pandas as pd
import json

from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec, load_validation_spec
from langchain_openai import ChatOpenAI

## Step 1: Load Target Schema and Normalization Spec

In [2]:
# Load the JSON Schema (used for both matching and normalization)
with open("data/target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec("data/target_schema.json")
target_columns = list(spec.columns.keys())

# Configure date column to parse year-only values (e.g., 1929, 2010)
# Note: actor_birthday will be constructed manually from month/day/year columns
spec.set_column("date", output_type="datetime", date_format="%Y")

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,title,string
2,date,datetime
3,studio,string
4,genre,string
5,budget,float
6,gross,float
7,oscar,bool
8,globe,bool
9,actor_name,string


## Step 2: Load Source Datasets

In [3]:
# Actors: tab-separated, no headers (pandas assigns integer column names)
df_actors = pd.read_csv(Path("data/actors.csv"), sep="\t", header=None, encoding="latin-1")
df_actors.attrs["dataset_name"] = "actors"
df_actors.head()

,0,1,2,3,4,5,6,7,8,9
0,f,1,1929,Janet Gaynor,7th Heaven,22,Pennsylvania,10,6,1906
1,f,2,1930,Mary Pickford,Coquette,37,Canada,4,8,1892
2,f,3,1931,Norma Shearer,The Divorcee,28,Canada,8,10,1902
3,f,4,1932,Marie Dressler,Min and Bill,63,Canada,11,9,1868
4,f,5,1933,Helen Hayes,The Sin of Madelon Claudet,32,Washington DC,10,10,1900


In [4]:
# Movies: has headers
df_movies = pd.read_csv(Path("data/movie_list.csv"))
df_movies.attrs["dataset_name"] = "movies"
df_movies.head()

,id,year,exclude,Film,Lead Studio,Rotten Tomatoes,Audience Score,Story,Genre,Number of Theatres in Opening Weekend (US),...,Foreign Gross,Worldwide Gross,Budget,Profit,Proftitability,Opening Weekend,Oscar,Bafta,Source,Column
0,1,2010,NaN,127 Hours,Independent,93.0,84,Escape,Adventure,916,...,42.4,60.73,18.0,42.73,337.39%,0.26,NaN,NaN,http://boxofficemojo.com/movies/?id=127hours.htm,NaN
1,2,2010,NaN,A Nightmare on Elm Street,Warner Bros.,13.0,40,Monster Force,Horror,3332,...,52.59,115.66,35.0,80.66,330.46%,32.9,NaN,NaN,NaN,NaN
2,3,2010,NaN,Alice in Wonderland,Disney,52.0,72,Journey And Return,Adventure,3728,...,690.2,1024.39,200.0,824.39,512.20%,116.1,NaN,NaN,NaN,NaN
3,4,2010,NaN,All About Steve,Independent,6.0,35,Comedy,Comedy,2251,...,6.26,40.13,15.0,25.13,267.53%,11.2,NaN,NaN,http://www.the-numbers.com/movies/2009/ABSTV.php,NaN
4,5,2010,y,All Good Things,Independent,33.0,64,The Riddle,Drama,2,...,0.062,0.64,20.0,-19.36,3.20%,0.037,NaN,NaN,http://www.wikipedia.org,NaN


## Step 3: LLM-Based Schema Matching

Pass `target_schema` so the LLM sees field descriptions (e.g., "Actor's birthday in ISO 8601 format").
This helps it make better matches, especially for ambiguous columns.

In [5]:
# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match actors dataset
actors_mapping = matcher.match(df_actors, df_target)

# Remove actor_birthday mappings - we'll construct it manually from columns 7, 8, 9
actors_mapping = actors_mapping[actors_mapping["target_column"] != "actor_birthday"]

actors_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,actors,2,target_schema,date,0.95,llm_based_matching
1,actors,3,target_schema,actor_name,0.95,llm_based_matching
2,actors,4,target_schema,title,0.95,llm_based_matching
3,actors,6,target_schema,actor_birthplace,0.95,llm_based_matching


In [6]:
# Match movies dataset
movies_mapping = matcher.match(df_movies, df_target)
movies_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,movies,id,target_schema,id,0.95,llm_based_matching
1,movies,year,target_schema,date,0.95,llm_based_matching
2,movies,Film,target_schema,title,0.95,llm_based_matching
3,movies,Lead Studio,target_schema,studio,0.95,llm_based_matching
4,movies,Genre,target_schema,genre,0.95,llm_based_matching
5,movies,Worldwide Gross,target_schema,gross,0.95,llm_based_matching
6,movies,Budget,target_schema,budget,0.95,llm_based_matching
7,movies,Oscar,target_schema,oscar,0.95,llm_based_matching


## Step 4: Translate and Normalize

`SchemaTranslator` renames columns and optionally applies normalization in one step.

In [7]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping
df_actors_normalized = translator.translate(
    df_actors, actors_mapping,
    normalize=spec, on_failure="keep"
)

# Construct actor_birthday from original columns: 7=month, 8=day, 9=year
df_actors_normalized["actor_birthday"] = pd.to_datetime(
    df_actors[9].astype(str) + "-" + df_actors[7].astype(str) + "-" + df_actors[8].astype(str),
    format="%Y-%m-%d",
    errors="coerce"
)

df_movies_normalized = translator.translate(
    df_movies, movies_mapping,
    normalize=spec, on_failure="keep"
)

In [8]:
# Show normalized actors (target columns only)
actor_cols = [c for c in target_columns if c in df_actors_normalized.columns]
df_actors_normalized[actor_cols].head(10)

,title,date,actor_name,actor_birthday,actor_birthplace
0,7th Heaven,1929-01-01,Janet Gaynor,1906-10-06,Pennsylvania
1,Coquette,1930-01-01,Mary Pickford,1892-04-08,Canada
2,The Divorcee,1931-01-01,Norma Shearer,1902-08-10,Canada
3,Min and Bill,1932-01-01,Marie Dressler,1868-11-09,Canada
4,The Sin of Madelon Claudet,1933-01-01,Helen Hayes,1900-10-10,Washington DC
5,Morning Glory,1934-01-01,Katharine Hepburn,1907-05-12,Connecticut
6,It Happened One Night,1935-01-01,Claudette Colbert,1903-09-13,France
7,Dangerous,1936-01-01,Bette Davis,1908-04-05,Massachusetts
8,The Great Zeigfeld,1937-01-01,Luise Rainer,1910-01-12,Germany
9,The Good Earth,1938-01-01,Luise Rainer,1910-01-12,Germany


In [9]:
# Show normalized movies (target columns only)
movie_cols = [c for c in target_columns if c in df_movies_normalized.columns]
df_movies_normalized[movie_cols].head(10)

,id,title,date,studio,genre,budget,gross,oscar
0,1,127 Hours,2010-01-01,Independent,Adventure,18.0,60.73,NaN
1,2,A Nightmare on Elm Street,2010-01-01,Warner Bros.,Horror,35.0,115.66,NaN
2,3,Alice in Wonderland,2010-01-01,Disney,Adventure,200.0,1024.39,NaN
3,4,All About Steve,2010-01-01,Independent,Comedy,15.0,40.13,NaN
4,5,All Good Things,2010-01-01,Independent,Drama,20.0,0.64,NaN
5,6,Alpha and Omega,2010-01-01,Crest,Animation,20.0,29.91,NaN
6,7,Barry Munday,2010-01-01,Independent,Comedy,NaN,NaN,NaN
7,8,Black Swan,2010-01-01,Fox,Drama,13.0,329.39,False
8,9,Brooklyn's Finest,2010-01-01,Independent,Action,17.0,36.31,NaN
9,10,Buried,2010-01-01,Independent,Drama,2.0,18.38,NaN


## Step 5: Validation Spec (Optional)

Validation constraints can be loaded separately from normalization.

In [10]:
val_spec = load_validation_spec("data/target_schema.json")
val_spec

{'id': {'nullable': False, 'type': 'string'},
 'title': {'nullable': False, 'type': 'string'},
 'date': {'nullable': False, 'type': 'string'},
 'studio': {'nullable': False, 'type': 'string'},
 'genre': {'nullable': False, 'type': 'string'},
 'budget': {'nullable': False, 'type': 'number', 'minimum': 0},
 'gross': {'nullable': False, 'type': 'number', 'minimum': 0},
 'oscar': {'nullable': False, 'type': 'boolean'},
 'globe': {'nullable': False, 'type': 'boolean'},
 'actor_name': {'nullable': False, 'type': 'string'},
 'actor_birthday': {'nullable': False, 'type': 'string'},
 'actor_birthplace': {'nullable': False, 'type': 'string'}}

## Summary

```python
# 1. Load JSON Schema
with open("target_schema.json") as f:
    target_schema = json.load(f)
spec = load_normalization_spec("target_schema.json")

# 2. Match source columns to target (with schema context)
matcher = LLMBasedSchemaMatcher(chat_model=llm, target_schema=target_schema)
mapping = matcher.match(df_source, df_target)

# 3. Translate + normalize
translator = SchemaTranslator()
df_normalized = translator.translate(df_source, mapping, normalize=spec)
```